# 01. Data Audit

**Purpose:** Profile the source datasets and identify data-quality issues before transformation.

**Inputs:** Source Parquet files under `BASE_PATH`.

**Outputs:** Audit findings and validation results. This notebook is diagnostic; it should not overwrite the Bronze/source data.

**Run order:** Run before the Silver transformation notebook. Confirm `BASE_PATH` points to the intended Databricks Volume.

In [ ]:
from pyspark.sql import functions as F


In [ ]:
BASE_PATH = "/Volumes/workspace/default/product_growth_data"

users = spark.read.parquet(f"{BASE_PATH}/users.parquet")
products = spark.read.parquet(f"{BASE_PATH}/products.parquet")
sessions = spark.read.parquet(f"{BASE_PATH}/sessions.parquet")
events = spark.read.parquet(f"{BASE_PATH}/events.parquet")
experiments = spark.read.parquet(f"{BASE_PATH}/experiments.parquet")
releases = spark.read.parquet(f"{BASE_PATH}/releases.parquet")

In [ ]:
tables = {
    "users": users,
    "products": products,
    "sessions": sessions,
    "events": events,
    "experiments": experiments,
    "releases": releases
}

for name, df in tables.items():
    print(f"{name:15} rows={df.count():,}  columns={len(df.columns)}")

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*70}")
    print(name.upper())
    df.printSchema()

In [ ]:
categorical_checks = {
    "users.device": users,
    "users.acquisition_channel": users,
    "users.customer_type": users,
    "events.event_name": events,
    "events.device": events,
    "events.acquisition_channel": events,
    "events.payment_status": events,
    "experiments.experiment_group": events
}

for label, df in categorical_checks.items():
    column = label.split(".")[1]

    print(f"\n{'='*60}")
    print(label.upper())

    df.groupBy(column) \
      .count() \
      .orderBy(F.desc("count")) \
      .show(50, truncate=False)

In [ ]:
key_checks = {
    "users.user_id": (users, "user_id"),
    "products.product_id": (products, "product_id"),
    "sessions.session_id": (sessions, "session_id"),
    "events.event_id": (events, "event_id")
}

for name, (df, col) in key_checks.items():
    total = df.count()
    unique = df.select(col).distinct().count()
    duplicate_rows = total - unique

    print(
        f"{name:25} "
        f"total={total:,} | "
        f"unique={unique:,} | "
        f"duplicate_rows={duplicate_rows:,}"
    )

In [ ]:
duplicate_event_ids = (
    events
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate event IDs:", duplicate_event_ids.count())

display(
    duplicate_event_ids
    .orderBy(F.desc("count"))
    .limit(20)
)

In [ ]:
duplicate_rows = (
    events
    .join(
        duplicate_event_ids.select("event_id"),
        on="event_id",
        how="inner"
    )
)

print("Rows belonging to duplicated event IDs:", duplicate_rows.count())

display(
    duplicate_rows
    .orderBy("event_id", "event_timestamp")
    .limit(50)
)

In [ ]:
duplicate_summary = (
    duplicate_rows
    .groupBy("event_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("event_timestamp").alias("timestamps"),
        F.countDistinct("user_id").alias("users"),
        F.countDistinct("session_id").alias("sessions"),
        F.countDistinct("event_name").alias("event_names"),
        F.countDistinct("product_id").alias("products"),
        F.countDistinct("device").alias("devices"),
        F.countDistinct("acquisition_channel").alias("channels")
    )
)

display(
    duplicate_summary
    .orderBy(
        F.desc("timestamps"),
        F.desc("devices")
    )
    .limit(50)
)

In [ ]:
conflicting_duplicates = duplicate_summary.filter(
    (F.col("timestamps") > 1) |
    (F.col("users") > 1) |
    (F.col("sessions") > 1) |
    (F.col("event_names") > 1) |
    (F.col("products") > 1) |
    (F.col("devices") > 1) |
    (F.col("channels") > 1)
)

print(
    "Duplicate event IDs with conflicting attributes:",
    conflicting_duplicates.count()
)

In [ ]:
display(
    duplicate_rows
    .groupBy("event_id")
    .agg(
        F.min("event_timestamp").alias("min_timestamp"),
        F.max("event_timestamp").alias("max_timestamp"),
        F.count("*").alias("rows")
    )
    .withColumn(
        "timestamp_diff_seconds",
        F.col("max_timestamp").cast("long") -
        F.col("min_timestamp").cast("long")
    )
    .filter(F.col("timestamp_diff_seconds") > 0)
    .orderBy(F.desc("timestamp_diff_seconds"))
    .limit(30)
)


In [ ]:
timestamp_gap_distribution = (
    duplicate_rows
    .groupBy("event_id")
    .agg(
        F.min("event_timestamp").alias("min_timestamp"),
        F.max("event_timestamp").alias("max_timestamp")
    )
    .withColumn(
        "gap_seconds",
        F.col("max_timestamp").cast("double") -
        F.col("min_timestamp").cast("double")
    )
)

display(
    timestamp_gap_distribution
    .groupBy("gap_seconds")
    .count()
    .orderBy("gap_seconds")
)

In [ ]:
print(
    "Duplicate IDs with timestamp gap:",
    timestamp_gap_distribution
    .filter(F.col("gap_seconds") > 0)
    .count()
)

print(
    "Duplicate IDs with gap > 10 seconds:",
    timestamp_gap_distribution
    .filter(F.col("gap_seconds") > 10)
    .count()
)

print(
    "Duplicate IDs with gap > 60 seconds:",
    timestamp_gap_distribution
    .filter(F.col("gap_seconds") > 60)
    .count()
)

In [ ]:
null_results = []

for name, df in tables.items():
    total = df.count()

    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()

        if null_count > 0:
            null_results.append((
                name,
                col,
                null_count,
                round(null_count / total * 100, 4)
            ))

null_df = spark.createDataFrame(
    null_results,
    ["table", "column", "null_count", "null_pct"]
)

display(
    null_df.orderBy(
        F.desc("null_pct")
    )
)

In [ ]:
important_events = [
    "product_view",
    "add_to_cart",
    "checkout_start",
    "payment_attempt",
    "purchase"
]

for event_type in important_events:

    subset = events.filter(F.col("event_name") == event_type)

    total = subset.count()

    product_nulls = subset.filter(
        F.col("product_id").isNull()
    ).count()

    price_nulls = subset.filter(
        F.col("price").isNull()
    ).count()

    device_nulls = subset.filter(
        F.col("device").isNull()
    ).count()

    print(
        f"{event_type:20} "
        f"total={total:,} | "
        f"product_id_null={product_nulls:,} | "
        f"price_null={price_nulls:,} | "
        f"device_null={device_nulls:,}"
    )

In [ ]:
product_view_missing = events.filter(
    (F.col("event_name") == "product_view") &
    F.col("product_id").isNull()
)

product_view_missing.groupBy("device").count().orderBy(
    F.desc("count")
).show()

product_view_missing.groupBy("acquisition_channel").count().orderBy(
    F.desc("count")
).show()

product_view_missing.groupBy(
    F.to_date("event_timestamp").alias("event_date")
).count().orderBy(
    "event_date"
).show(20)

In [ ]:
product_view_price_missing = events.filter(
    (F.col("event_name") == "product_view") &
    F.col("price").isNull()
)

product_view_price_missing.groupBy("device").count().orderBy(
    F.desc("count")
).show()

product_view_price_missing.groupBy("acquisition_channel").count().orderBy(
    F.desc("count")
).show()

product_view_price_missing.groupBy(
    F.to_date("event_timestamp").alias("event_date")
).count().orderBy(
    "event_date"
).show(20)

In [ ]:
purchase_device_missing = events.filter(
    (F.col("event_name") == "purchase") &
    F.col("device").isNull()
)

purchase_device_missing.select(
    "event_id",
    "user_id",
    "session_id",
    "event_timestamp"
).join(
    sessions.select(
        "session_id",
        F.col("device").alias("session_device")
    ),
    on="session_id",
    how="left"
).groupBy(
    "session_device"
).count().orderBy(
    F.desc("count")
).show()

In [ ]:
invalid_product_events = events.filter(
    F.col("product_id").isNotNull()
).join(
    products.select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Invalid product references:", invalid_product_events.count())

invalid_product_events.groupBy(
    "event_name"
).count().orderBy(
    F.desc("count")
).show()

In [ ]:
invalid_product_events.select(
    "product_id"
).distinct().count()

In [ ]:
invalid_product_events.groupBy(
    "product_id"
).count().orderBy(
    F.desc("count")
).show(20)

In [ ]:
products.select(
    F.min("product_id").alias("min_product_id"),
    F.max("product_id").alias("max_product_id")
).show()
invalid_product_events.select(
    F.min("product_id").alias("min_invalid_product_id"),
    F.max("product_id").alias("max_invalid_product_id")
).show()

In [ ]:
from pyspark.sql import functions as F

events.filter(
    F.col("price") <= 0
).groupBy(
    "event_name"
).count().orderBy(
    F.desc("count")
).show()

In [ ]:
events.filter(
    F.col("quantity") <= 0
).groupBy(
    "event_name"
).count().orderBy(
    F.desc("count")
).show()

In [ ]:
events.filter(
    F.col("event_name") == "purchase"
).groupBy(
    "payment_status"
).count().orderBy(
    F.desc("count")
).show()

In [ ]:
events.filter(
    F.col("device").isNull()
).join(
    sessions.select(
        "session_id",
        F.col("device").alias("session_device")
    ),
    on="session_id",
    how="left"
).groupBy(
    "session_device"
).count().orderBy(
    F.desc("count")
).show()